# Build or restore the final search index

Use a fresh Colab T4 runtime. Upload the supplied `pjt` folder to MyDrive/pjt first. This uses the final corpus and does not retrain BERT. The deployment notebook also runs these same steps.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess
from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/pjt')
LOCAL_DIR = Path('/content/pjt')
if not (DRIVE_DIR / 'data/bert_failure_classifier_runtime_v4/model.safetensors').is_file():
    raise FileNotFoundError('Extract the supplied ZIP first, then upload its pjt folder to MyDrive/pjt, including data/.')
shutil.copytree(DRIVE_DIR, LOCAL_DIR, dirs_exist_ok=True,
    ignore=shutil.ignore_patterns('node_modules', '__pycache__', '.git', '.venv'))
os.chdir(LOCAL_DIR)
if str(LOCAL_DIR) not in sys.path:
    sys.path.insert(0, str(LOCAL_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('Working project:', LOCAL_DIR)


In [ ]:
from src.prepare_deployment import prepare, check_assets
prepare()

# Persist both indices so later Colab sessions reuse them.
for name in ['faiss_index', 'faiss_index_with_source_passages']:
    source = LOCAL_DIR / 'data' / name
    if source.is_dir():
        shutil.copytree(source, DRIVE_DIR / 'data' / name, dirs_exist_ok=True)
print(check_assets())
print('Index cache saved in your project data folder on Drive.')
